# 📖 Notebook 2: Auction Lifecycle Management

An auction isn't just about bids. It has a **lifecycle** — it's created, goes live, accepts bids, and eventually ends. Managing these transitions correctly is important for data integrity and user trust.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to create an auction with proper validation
- The auction state machine: `active` → `ended` / `cancelled`
- How to find and close expired auctions (simulating a cron job)
- Fault-tolerant auction ending using database transactions
- Dynamic end times — extending auctions on late bids ("anti-sniping")

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/online-auction
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `auction_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "auction_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

---
## 📝 Creating an Auction

To create an auction, we need:
1. An **item** to sell (name, description)
2. A **starting price** (minimum bid)
3. A **start date** and **end date**

In our database, items and auctions are separate tables. This means:
- An item can be **relisted** if it doesn't sell
- Item details can be updated without touching auction logic
- We can add item search/categories later

Let's build a function to create an auction from scratch.

In [ ]:
def create_auction(seller_id, item_name, item_description, starting_price, duration_days=7):
    """
    Create a new auction. This:
    1. Creates the item record
    2. Creates the auction record linked to the item
    3. Uses a transaction so both succeed or both fail
    """
    # Validate inputs
    if starting_price <= 0:
        raise ValueError("Starting price must be positive")
    if duration_days <= 0:
        raise ValueError("Duration must be at least 1 day")
    if not item_name.strip():
        raise ValueError("Item name cannot be empty")

    conn = get_db_connection()
    cur = conn.cursor()

    try:
        # Create the item
        cur.execute(
            "INSERT INTO items (seller_id, name, description) VALUES (%s, %s, %s) RETURNING id",
            (seller_id, item_name, item_description)
        )
        item_id = cur.fetchone()[0]

        # Create the auction
        start_date = datetime.now()
        end_date = start_date + timedelta(days=duration_days)

        cur.execute(
            """INSERT INTO auctions (item_id, seller_id, starting_price, max_bid_amount,
                                     start_date, end_date, status)
               VALUES (%s, %s, %s, %s, %s, %s, 'active') RETURNING id""",
            (item_id, seller_id, starting_price, starting_price, start_date, end_date)
        )
        auction_id = cur.fetchone()[0]

        # Both succeeded — commit the transaction
        conn.commit()
        conn.close()

        return {
            "auction_id": auction_id,
            "item_id": item_id,
            "item_name": item_name,
            "starting_price": starting_price,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "status": "active"
        }
    except Exception as e:
        conn.rollback()
        conn.close()
        raise e

# Create a test auction
auction = create_auction(
    seller_id=1,
    item_name="Vintage Polaroid Camera",
    item_description="1970s Polaroid SX-70, fully functional with original leather case.",
    starting_price=150.00,
    duration_days=5
)

print("🎉 Auction created!")
for key, value in auction.items():
    print(f"   {key}: {value}")

---
## 🔄 The Auction State Machine

An auction has three possible states:

```
                ┌──────────┐
  create ──────►│  active  │
                └────┬─────┘
                     │
            ┌────────┼────────┐
            ▼                 ▼
      ┌──────────┐     ┌───────────┐
      │  ended   │     │ cancelled │
      └──────────┘     └───────────┘
```

**Rules:**
- Only `active` auctions accept bids
- An auction transitions to `ended` when `end_date` passes
- A seller can `cancel` an auction **only if no bids have been placed**
- Once `ended` or `cancelled`, an auction cannot go back to `active`

Let's implement these state transitions.

In [ ]:
def get_auction_details(auction_id):
    """Fetch full auction details including item info and bid count."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute(
        """SELECT a.id, i.name, a.starting_price, a.max_bid_amount,
                  a.max_bid_user_id, a.start_date, a.end_date, a.status,
                  (SELECT COUNT(*) FROM bids b WHERE b.auction_id = a.id AND b.status = 'accepted') as bid_count
           FROM auctions a
           JOIN items i ON a.item_id = i.id
           WHERE a.id = %s""",
        (auction_id,)
    )
    row = cur.fetchone()
    conn.close()

    if not row:
        return None

    return {
        "auction_id": row[0],
        "item_name": row[1],
        "starting_price": float(row[2]),
        "max_bid_amount": float(row[3]),
        "max_bid_user_id": row[4],
        "start_date": row[5],
        "end_date": row[6],
        "status": row[7],
        "bid_count": row[8]
    }

def display_auction(details):
    """Pretty-print auction details."""
    if not details:
        print("❌ Auction not found")
        return

    status_emoji = {"active": "🟢", "ended": "🔴", "cancelled": "⚪"}
    emoji = status_emoji.get(details["status"], "❓")

    print(f"{emoji} Auction #{details['auction_id']}: {details['item_name']}")
    print(f"   Status:        {details['status']}")
    print(f"   Starting at:   ${details['starting_price']:,.2f}")
    print(f"   Current max:   ${details['max_bid_amount']:,.2f}")
    print(f"   Leading bidder: User {details['max_bid_user_id'] or 'None'}")
    print(f"   Total bids:    {details['bid_count']}")
    print(f"   Ends:          {details['end_date']}")

# View some auctions
print("📋 Active auctions from our seed data:\n")
for aid in [1, 7, 9]:
    display_auction(get_auction_details(aid))
    print()

In [ ]:
def cancel_auction(auction_id, seller_id):
    """
    Cancel an auction. Only works if:
    1. The auction is still 'active'
    2. The caller is the seller
    3. No bids have been placed
    """
    conn = get_db_connection()
    cur = conn.cursor()

    try:
        # Lock the auction row to prevent race conditions during cancel
        cur.execute(
            "SELECT status, seller_id, max_bid_user_id FROM auctions WHERE id = %s FOR UPDATE",
            (auction_id,)
        )
        row = cur.fetchone()

        if not row:
            conn.rollback()
            conn.close()
            return {"success": False, "reason": "Auction not found"}

        status, owner_id, max_bid_user = row

        if owner_id != seller_id:
            conn.rollback()
            conn.close()
            return {"success": False, "reason": "Only the seller can cancel"}

        if status != "active":
            conn.rollback()
            conn.close()
            return {"success": False, "reason": f"Cannot cancel — auction is already '{status}'"}

        if max_bid_user is not None:
            conn.rollback()
            conn.close()
            return {"success": False, "reason": "Cannot cancel — bids have been placed"}

        # All checks passed — cancel it
        cur.execute(
            "UPDATE auctions SET status = 'cancelled', updated_at = NOW() WHERE id = %s",
            (auction_id,)
        )
        conn.commit()
        conn.close()
        return {"success": True, "reason": "Auction cancelled"}

    except Exception as e:
        conn.rollback()
        conn.close()
        return {"success": False, "reason": str(e)}

# Test cancellation scenarios
print("🧪 Testing auction cancellation:\n")

# Auction 6 has no bids (max_bid_user_id is NULL), seller is user 6
result = cancel_auction(6, seller_id=6)
print(f"Cancel auction 6 (no bids, correct seller):")
print(f"   Result: {result}")
print()

# Auction 1 has bids — should fail
result = cancel_auction(1, seller_id=1)
print(f"Cancel auction 1 (has bids):")
print(f"   Result: {result}")
print()

# Try to cancel auction 1 as wrong seller — should fail
result = cancel_auction(1, seller_id=99)
print(f"Cancel auction 1 (wrong seller):")
print(f"   Result: {result}")

---
## ⏰ Ending Expired Auctions

Auctions don't end by themselves. We need a process that periodically checks for auctions whose `end_date` has passed and transitions them to `ended`.

In production, this would be a **cron job** or a **scheduled task**. Here we'll simulate it.

### The Process
1. Find all `active` auctions where `end_date < NOW()`
2. For each one, set `status = 'ended'`
3. Log who won (the `max_bid_user_id`)

### Why Not End On the Exact Second?
Checking every auction's end time at the exact second would require complex timer infrastructure. A cron job that runs every 30–60 seconds is simpler and good enough for most cases. The auction will end at most 60 seconds late — acceptable for the vast majority of auctions.

In [ ]:
def close_expired_auctions():
    """
    Find and close all auctions whose end_date has passed.
    This simulates what a cron job would do every 30-60 seconds.
    Returns a list of auctions that were closed.
    """
    conn = get_db_connection()
    cur = conn.cursor()

    # Find expired auctions
    cur.execute(
        """SELECT a.id, i.name, a.max_bid_amount, a.max_bid_user_id
           FROM auctions a
           JOIN items i ON a.item_id = i.id
           WHERE a.status = 'active' AND a.end_date < NOW()"""
    )
    expired = cur.fetchall()

    closed = []
    for auction_id, item_name, max_bid, winner_id in expired:
        cur.execute(
            "UPDATE auctions SET status = 'ended', updated_at = NOW() WHERE id = %s AND status = 'active'",
            (auction_id,)
        )
        if cur.rowcount == 1:
            closed.append({
                "auction_id": auction_id,
                "item_name": item_name,
                "final_price": float(max_bid),
                "winner_id": winner_id
            })

    conn.commit()
    conn.close()
    return closed

# Run the closer
print("⏰ Running auction closer (simulating cron job)...\n")
closed = close_expired_auctions()

if closed:
    print(f"🔴 Closed {len(closed)} expired auction(s):\n")
    for a in closed:
        winner = f"User {a['winner_id']}" if a['winner_id'] else "No winner (no bids)"
        print(f"   Auction #{a['auction_id']}: {a['item_name']}")
        print(f"   Final price: ${a['final_price']:,.2f} — Winner: {winner}")
        print()
else:
    print("✅ No expired auctions found (all end dates are in the future)")

In [ ]:
# Let's create an auction that expires immediately and then close it

conn = get_db_connection()
cur = conn.cursor()

# Create an item
cur.execute(
    "INSERT INTO items (seller_id, name, description) VALUES (%s, %s, %s) RETURNING id",
    (5, "Test Widget", "A widget that expires immediately for testing")
)
item_id = cur.fetchone()[0]

# Create an auction that ended 1 minute ago
cur.execute(
    """INSERT INTO auctions (item_id, seller_id, starting_price, max_bid_amount,
                             start_date, end_date, status)
       VALUES (%s, %s, 100.00, 250.00, NOW() - interval '2 days', NOW() - interval '1 minute', 'active')
       RETURNING id""",
    (item_id, 5)
)
test_auction_id = cur.fetchone()[0]

# Add a fake winning bid
cur.execute(
    "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, 42, 250.00, 'accepted')",
    (test_auction_id,)
)
cur.execute(
    "UPDATE auctions SET max_bid_user_id = 42 WHERE id = %s",
    (test_auction_id,)
)
conn.commit()
conn.close()

print(f"Created test auction #{test_auction_id} that expired 1 minute ago.")
print(f"Current state:")
display_auction(get_auction_details(test_auction_id))
print()

# Now run the closer
print("⏰ Running auction closer...\n")
closed = close_expired_auctions()
for a in closed:
    winner = f"User {a['winner_id']}" if a['winner_id'] else "No winner"
    print(f"   🔴 Closed Auction #{a['auction_id']}: {a['item_name']}")
    print(f"      Final price: ${a['final_price']:,.2f} — Winner: {winner}")

print()
print(f"After closing:")
display_auction(get_auction_details(test_auction_id))

---
## 🏗️ Fault-Tolerant Auction Ending

What if our closer crashes halfway through? We might close some auctions and miss others. Or worse, we might update the status but fail to record the winner.

### The Problem
```
Closer starts
 → Finds 5 expired auctions
 → Closes auction #1 ✅
 → Closes auction #2 ✅
 → CRASH! 💥
 → Auctions #3, #4, #5 are still 'active' but expired
```

### The Fix: Idempotent Operations
Our closer is already **idempotent** — running it twice produces the same result. The `WHERE status = 'active'` clause means we skip auctions that were already closed. So if it crashes and restarts, it simply picks up where it left off.

Let's verify this.

In [ ]:
# Verify idempotency: running the closer twice should be safe

# Create another expired auction
conn = get_db_connection()
cur = conn.cursor()
cur.execute(
    "INSERT INTO items (seller_id, name, description) VALUES (%s, %s, %s) RETURNING id",
    (3, "Another Widget", "Testing idempotency")
)
item_id = cur.fetchone()[0]
cur.execute(
    """INSERT INTO auctions (item_id, seller_id, starting_price, max_bid_amount,
                             start_date, end_date, status)
       VALUES (%s, %s, 50.00, 50.00, NOW() - interval '1 day', NOW() - interval '30 seconds', 'active')
       RETURNING id""",
    (item_id, 3)
)
idempotent_auction_id = cur.fetchone()[0]
conn.commit()
conn.close()

print(f"Created expired auction #{idempotent_auction_id}\n")

# Run closer twice
print("Run 1:")
closed_1 = close_expired_auctions()
print(f"   Closed {len(closed_1)} auction(s)")

print("\nRun 2 (should be idempotent — nothing new to close):")
closed_2 = close_expired_auctions()
print(f"   Closed {len(closed_2)} auction(s)")

print("\n✅ Running the closer twice is safe. Already-ended auctions are skipped.")

---
## 🔄 Dynamic End Times (Anti-Sniping)

**Sniping** is when someone places a bid in the last second of an auction, giving other bidders no time to respond. eBay suffers from this.

Many auction platforms solve this with **dynamic end times**: if a bid arrives in the last N minutes, the auction extends by N more minutes. This keeps the auction going as long as people are actively bidding.

### How It Works
1. When a bid is placed, check if `end_date` is within the next 5 minutes
2. If yes, extend `end_date` by 5 minutes from now
3. This means the auction only truly ends when 5 minutes pass with no new bids

In [ ]:
ANTI_SNIPE_WINDOW_MINUTES = 5

def place_bid_with_extension(auction_id, user_id, amount):
    """
    Place a bid using row-level locking, with anti-sniping: if the auction
    ends within N minutes, extend the end_date by N more minutes.

    Important: we do the time arithmetic INSIDE Postgres using NOW() and
    INTERVAL. This avoids any mismatch between the Python process's local
    timezone and the Postgres server timezone (a subtle bug when your app
    server and DB run in different timezones — very common in containerized
    setups where Python uses local time and Postgres runs in UTC).
    """
    conn = get_db_connection()
    cur = conn.cursor()

    try:
        cur.execute(
            """SELECT max_bid_amount, end_date, status,
                      EXTRACT(EPOCH FROM (end_date - NOW())) AS seconds_remaining
               FROM auctions WHERE id = %s FOR UPDATE""",
            (auction_id,)
        )
        row = cur.fetchone()
        if not row:
            conn.rollback()
            conn.close()
            return {"status": "error", "reason": "Auction not found"}

        current_max = float(row[0])
        end_date = row[1]
        status = row[2]
        seconds_remaining = float(row[3]) if row[3] is not None else 0.0

        if status != "active":
            conn.rollback()
            conn.close()
            return {"status": "rejected", "reason": f"Auction is {status}"}

        if seconds_remaining <= 0:
            conn.rollback()
            conn.close()
            return {"status": "rejected", "reason": "Auction has already ended"}

        if amount <= current_max:
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
                (auction_id, user_id, amount)
            )
            conn.commit()
            conn.close()
            return {"status": "rejected", "reason": f"Bid ${amount} is not higher than ${current_max}"}

        cur.execute(
            "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
            (auction_id, user_id, amount)
        )

        # Anti-sniping: if we're inside the window, extend end_date.
        # The CASE is evaluated by Postgres using its own NOW(), so the
        # Python timezone is irrelevant.
        threshold = f"{ANTI_SNIPE_WINDOW_MINUTES} minutes"
        cur.execute(
            f"""UPDATE auctions
                SET max_bid_amount = %s,
                    max_bid_user_id = %s,
                    end_date = CASE
                        WHEN end_date - NOW() < INTERVAL '{threshold}'
                        THEN NOW() + INTERVAL '{threshold}'
                        ELSE end_date
                    END,
                    updated_at = NOW()
                WHERE id = %s
                RETURNING end_date""",
            (amount, user_id, auction_id)
        )
        new_end_date = cur.fetchone()[0]

        conn.commit()
        conn.close()

        extended = new_end_date != end_date
        result = {
            "status": "accepted",
            "amount": amount,
            "extended": extended,
            "old_end_date": end_date.isoformat(),
            "new_end_date": new_end_date.isoformat(),
        }
        if extended:
            result["extra_time"] = f"{ANTI_SNIPE_WINDOW_MINUTES} minutes"
        return result

    except Exception as e:
        conn.rollback()
        conn.close()
        return {"status": "error", "reason": str(e)}


In [ ]:
# Demo: Create an auction that ends in 2 minutes, then snipe it

conn = get_db_connection()
cur = conn.cursor()

cur.execute(
    "INSERT INTO items (seller_id, name, description) VALUES (%s, %s, %s) RETURNING id",
    (8, "Snipe Test Item", "This auction ends soon!")
)
item_id = cur.fetchone()[0]

# Auction ends in 2 minutes — inside the 5-minute anti-snipe window.
# We use Postgres NOW() so there's no client/server timezone confusion.
cur.execute(
    """INSERT INTO auctions (item_id, seller_id, starting_price, max_bid_amount,
                             start_date, end_date, status)
       VALUES (%s, %s, 100.00, 100.00, NOW() - interval '6 days',
               NOW() + interval '2 minutes', 'active')
       RETURNING id""",
    (item_id, 8)
)
snipe_auction_id = cur.fetchone()[0]
conn.commit()
conn.close()

print(f"🎯 Created auction #{snipe_auction_id} ending in 2 minutes\n")

print("Before bid:")
details = get_auction_details(snipe_auction_id)
display_auction(details)
print(f"   ⏱️  Original end: {details['end_date']}")

# Place a last-second bid
print("\n💥 Placing a last-minute bid of $200...\n")
result = place_bid_with_extension(snipe_auction_id, user_id=25, amount=200.00)
print(f"   Result: {result}")

print("\nAfter bid:")
details = get_auction_details(snipe_auction_id)
display_auction(details)
print(f"   ⏱️  New end:      {details['end_date']}")

if result.get("extended"):
    print("\n🛡️  The auction was extended by 5 minutes to prevent sniping!")
    print("    This gives other bidders a fair chance to respond.")
else:
    print("\n⚠️  Auction was NOT extended — the bid didn't land inside the")
    print("    anti-snipe window. Try with a smaller time-remaining in the INSERT.")


---
## 📊 Viewing Active Auctions

In a real system, users browse and search auctions. Let's build a simple listing function that shows:
- Active auctions sorted by end date (ending soonest first)
- Each auction's current price and bid count

In [ ]:
def list_active_auctions(limit=10):
    """List active auctions, ending soonest first."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute(
        """SELECT a.id, i.name, a.starting_price, a.max_bid_amount,
                  a.end_date,
                  (SELECT COUNT(*) FROM bids b WHERE b.auction_id = a.id AND b.status = 'accepted') as bid_count
           FROM auctions a
           JOIN items i ON a.item_id = i.id
           WHERE a.status = 'active'
           ORDER BY a.end_date ASC
           LIMIT %s""",
        (limit,)
    )
    rows = cur.fetchall()
    conn.close()

    print(f"{'ID':>4}  {'Item':<40}  {'Start':>10}  {'Current':>10}  {'Bids':>5}  {'Ends'}")
    print("-" * 110)
    for row in rows:
        name = row[1][:38] + ".." if len(row[1]) > 40 else row[1]
        ends = row[4].strftime("%Y-%m-%d %H:%M")
        print(f"{row[0]:>4}  {name:<40}  ${float(row[2]):>9,.2f}  ${float(row[3]):>9,.2f}  {row[5]:>5}  {ends}")

print("📋 Active auctions (ending soonest first):\n")
list_active_auctions()

---
## 🧠 Summary

### Auction Lifecycle
| State | Accepts Bids? | Can Cancel? | Transitions To |
|-------|:---:|:---:|---|
| `active` | ✅ | ✅ (if no bids) | `ended`, `cancelled` |
| `ended` | ❌ | ❌ | — |
| `cancelled` | ❌ | ❌ | — |

### Key Patterns
- **Transactions** — Creating an auction (item + auction) must be atomic
- **Idempotent closers** — The cron job can crash and restart safely
- **Anti-sniping** — Extend the end time when late bids arrive
- **Validation** — Check auction status before accepting bids or cancellations

### In Production
- Use a **message queue** (Kafka) to process bids durably
- Use a **scheduled task system** (cron, Celery, AWS Step Functions) to close auctions
- Send **email/push notifications** to the winner when an auction ends
- Implement a **payment flow** after the auction closes

### What's Next
- **Notebook 3**: Real-time bid notifications — pushing updates to all watchers using Redis Pub/Sub